# Match Outcome Prediction — Elo and Grid Search

> **Research project:** Conducted in collaboration with Professor Piotr Duch of Lodz University of Technology (TUL), extending his English Premier League match outcome prediction research with additional autoencoder experiments.


## 1. Setup


In [ ]:
import warnings

warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, classification_report, confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, TimeSeriesSplit,
)
from sklearn.pipeline import Pipeline
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DATASET_MODE = 'home_away_form_3'      # 'match_form_5' or 'home_away_form_3'
SPLIT_MODE = 'stratified_80_20'         # 'last_season', 'chronological_80_20', 'stratified_80_20'
TEST_SEASON = '2023/24'
USE_CLASS_WEIGHTS = False
USE_ELO = True                 # True: include Elo; False: exclude Elo from every model
USE_GRID_SEARCH = True         # True: tune classical models using training data only
GRID_SEARCH_SCORING = 'accuracy'  # 'accuracy', 'f1_macro', or 'balanced_accuracy'
GRID_CV_FOLDS = 5
SEED = 42

ROLLING_WINDOW = 5 if DATASET_MODE == 'match_form_5' else 3

AE_EPOCHS = 100
AE_LR = 1e-3
AE_BATCH = 64
LATENT_DIM = 16

ELO_INITIAL = 1500.0
ELO_K = 20.0
ELO_HOME_ADVANTAGE = 65.0

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
print(f'Dataset: {DATASET_MODE} | window={ROLLING_WINDOW}')
print(
    f'Split: {SPLIT_MODE} | class weights={USE_CLASS_WEIGHTS} | '
    f'Elo={USE_ELO} | grid search={USE_GRID_SEARCH} | '
    f'grid scoring={GRID_SEARCH_SCORING}'
)


## 2. Data Upload


In [ ]:
from google.colab import files

print("Upload match.csv, league.csv, league_season.csv and vaastav_stats.csv")
files.upload()

match = pd.read_csv('match.csv')
league = pd.read_csv('league.csv')
league_season = pd.read_csv('league_season.csv')
vaastav_stats = pd.read_csv('vaastav_stats.csv')

print("Loaded CSV files successfully")

## 3. Raw Data Preparation


In [ ]:
print("Loading CSV tables...")

vs = vaastav_stats.copy()

seasons = league_season.copy()

# Attach competition names and keep the Premier League seasons explicitly.
if {'league_id', 'league_name'}.issubset(league.columns) and 'league_id' in seasons.columns:
    league_lookup = league[['league_id', 'league_name']].drop_duplicates('league_id')
    seasons = seasons.merge(league_lookup, on='league_id', how='left')
    seasons = seasons[seasons['league_name'].str.casefold().eq('premier league')].copy()
else:
    raise ValueError('league.csv must contain league_id and league_name, and league_season.csv must contain league_id')

if 'start_date' in seasons.columns and 'end_date' in seasons.columns:
    seasons['season_name'] = (
        pd.to_datetime(seasons['start_date']).dt.year.astype(str)
        + '/'
        + pd.to_datetime(seasons['end_date']).dt.year.astype(str).str[-2:]
    )

print(f"Match cols: {match.columns.tolist()}")
print(f"Vaastav cols: {vs.columns.tolist()}")


## 4. Target Construction


In [ ]:
def find_col(df, *hints):
    for h in hints:
        if h in df.columns:
            return h
    for h in hints:
        for c in df.columns:
            if h.lower() in c.lower():
                return c
    return None

match_id_col  = find_col(match, 'match_id')
date_col      = find_col(match, 'kickoff_time', 'match_date', 'date')
home_team_col = find_col(match, 'home_team_id', 'team_h', 'home_team')
away_team_col = find_col(match, 'away_team_id', 'team_a', 'away_team')
ls_col        = find_col(match, 'league_season_id')
was_home_col  = find_col(vs, 'was_home', 'is_home')
h_score_col   = find_col(vs, 'team_h_score', 'home_score')
a_score_col   = find_col(vs, 'team_a_score', 'away_score')

required = {
    'match id': match_id_col, 'date': date_col, 'home team': home_team_col,
    'away team': away_team_col, 'season': ls_col, 'was_home': was_home_col,
    'home score': h_score_col, 'away score': a_score_col,
}
missing = [name for name, value in required.items() if value is None]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

matches = match[[match_id_col, date_col, home_team_col, away_team_col, ls_col]].copy()
matches.columns = ['match_id', 'match_date', 'home_team_id', 'away_team_id', 'league_season_id']
matches['match_date'] = pd.to_datetime(matches['match_date'], utc=True).dt.tz_localize(None)
matches = matches.merge(seasons, on='league_season_id', how='left')

scores = (
    vs[['match_id', h_score_col, a_score_col]]
    .dropna()
    .drop_duplicates('match_id')
    .rename(columns={h_score_col: 'home_score', a_score_col: 'away_score'})
)
scores[['home_score', 'away_score']] = scores[['home_score', 'away_score']].apply(
    pd.to_numeric, errors='coerce'
)
scores = scores.dropna(subset=['home_score', 'away_score'])
scores['outcome'] = np.select(
    [scores.home_score > scores.away_score, scores.home_score < scores.away_score],
    ['H', 'A'], default='D'
)

matches = matches.merge(scores, on='match_id', how='inner')
matches = matches.sort_values(['match_date', 'match_id']).reset_index(drop=True)

print(f'Matches with outcomes: {len(matches):,}')
print(matches['outcome'].value_counts().to_string())
assert matches['match_id'].is_unique


## 5. Team-Level Feature Engineering


In [ ]:
# Additive player statistics: summing these across players has a valid meaning.
ADDITIVE_PLAYER_COLS = [
    'assists', 'own_goals', 'penalties_saved', 'penalties_missed',
    'yellow_cards', 'red_cards', 'saves',
    'influence', 'creativity', 'threat',
]

missing_additive = [c for c in ADDITIVE_PLAYER_COLS if c not in vs.columns]
if missing_additive:
    raise ValueError(f'Vaastav file is missing expected columns: {missing_additive}')

for c in ADDITIVE_PLAYER_COLS:
    vs[c] = pd.to_numeric(vs[c], errors='coerce').fillna(0.0)

player_team = (
    vs[['match_id', was_home_col] + ADDITIVE_PLAYER_COLS]
    .groupby(['match_id', was_home_col], as_index=False)[ADDITIVE_PLAYER_COLS]
    .sum()
    .rename(columns={was_home_col: 'was_home'})
)

# Two side rows per match with authoritative team-level score features.
home_side = matches[[
    'match_id', 'match_date', 'home_team_id', 'away_team_id',
    'home_score', 'away_score', 'outcome'
]].copy()
home_side = home_side.rename(columns={
    'home_team_id': 'team_id', 'away_team_id': 'opponent_id',
    'home_score': 'goals_scored', 'away_score': 'goals_conceded',
})
home_side['was_home'] = 1

away_side = matches[[
    'match_id', 'match_date', 'home_team_id', 'away_team_id',
    'home_score', 'away_score', 'outcome'
]].copy()
away_side = away_side.rename(columns={
    'away_team_id': 'team_id', 'home_team_id': 'opponent_id',
    'away_score': 'goals_scored', 'home_score': 'goals_conceded',
})
away_side['was_home'] = 0

sides = pd.concat([home_side, away_side], ignore_index=True)
sides['clean_sheets'] = (sides['goals_conceded'] == 0).astype(float)

team_match = sides.merge(player_team, on=['match_id', 'was_home'], how='left')
team_match[ADDITIVE_PLAYER_COLS] = team_match[ADDITIVE_PLAYER_COLS].fillna(0.0)
team_match = team_match.sort_values(['match_date', 'match_id', 'was_home']).reset_index(drop=True)

safe_cols = [
    'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
    'own_goals', 'penalties_saved', 'penalties_missed',
    'yellow_cards', 'red_cards', 'saves',
    'influence', 'creativity', 'threat',
]

assert not team_match.duplicated(['match_id', 'was_home']).any()
assert set(team_match['clean_sheets'].dropna().unique()).issubset({0.0, 1.0})
assert (team_match['goals_conceded'] >= 0).all()

print(f'Team-match rows: {len(team_match):,}')
print(f'Correct team features ({len(safe_cols)}): {safe_cols}')
print('Removed: minutes (near-constant), ict_index (redundant), player-level goals_conceded/clean_sheets sums.')


### 5.1 Elo Ratings


In [ ]:
# Optional pre-match Elo calculated chronologically.
# When USE_ELO=False, Elo columns are created only as placeholders and are
# excluded from FEAT_COLS, so every retained model runs without Elo.

if USE_ELO:
    ratings = {}
    elo_rows = []

    for r in matches.sort_values(['match_date', 'match_id']).itertuples(index=False):
        rh = ratings.get(r.home_team_id, ELO_INITIAL)
        ra = ratings.get(r.away_team_id, ELO_INITIAL)

        expected_h = 1.0 / (
            1.0 + 10.0 ** ((ra - (rh + ELO_HOME_ADVANTAGE)) / 400.0)
        )
        actual_h = 1.0 if r.outcome == 'H' else (0.5 if r.outcome == 'D' else 0.0)

        # Store ratings BEFORE applying the current match result.
        elo_rows.append({
            'match_id': r.match_id,
            'home_elo': rh,
            'away_elo': ra,
        })

        delta = ELO_K * (actual_h - expected_h)
        ratings[r.home_team_id] = rh + delta
        ratings[r.away_team_id] = ra - delta

    elo_match = pd.DataFrame(elo_rows)

    home_elo_side = elo_match.merge(
        matches[['match_id', 'home_team_id', 'away_team_id']],
        on='match_id',
        validate='one_to_one',
    ).rename(columns={
        'home_team_id': 'team_id',
        'away_team_id': 'opponent_id',
        'home_elo': 'team_elo',
    })[['match_id', 'team_id', 'opponent_id', 'team_elo']]

    away_elo_side = elo_match.merge(
        matches[['match_id', 'home_team_id', 'away_team_id']],
        on='match_id',
        validate='one_to_one',
    ).rename(columns={
        'away_team_id': 'team_id',
        'home_team_id': 'opponent_id',
        'away_elo': 'team_elo',
    })[['match_id', 'team_id', 'opponent_id', 'team_elo']]

    team_match = team_match.merge(
        pd.concat([home_elo_side, away_elo_side], ignore_index=True),
        on=['match_id', 'team_id', 'opponent_id'],
        how='left',
        validate='one_to_one',
    )

    assert team_match['team_elo'].notna().all()
    print(elo_match.head().to_string(index=False))
    print('Elo enabled: every row stores ratings before that match result is applied.')
else:
    team_match['team_elo'] = np.nan
    print('Elo disabled: home_elo and away_elo will not be included in FEAT_COLS.')


### 5.2 Rolling Form Features


In [ ]:
print(f'Building {DATASET_MODE} with a {ROLLING_WINDOW}-match window...')

ordered = team_match.sort_values(['team_id', 'match_date', 'match_id']).copy()
roll_cols = []

if DATASET_MODE == 'match_form_5':
    group_keys = ['team_id']
elif DATASET_MODE == 'home_away_form_3':
    # Home team: previous home matches only. Away team: previous away matches only.
    group_keys = ['team_id', 'was_home']
else:
    raise ValueError("DATASET_MODE must be 'match_form_5' or 'home_away_form_3'")

for c in safe_cols:
    rc = f'roll_{c}'
    ordered[rc] = ordered.groupby(group_keys, sort=False)[c].transform(
        lambda s: s.shift(1).rolling(
            ROLLING_WINDOW,
            min_periods=ROLLING_WINDOW,
        ).mean()
    )
    roll_cols.append(rc)

# No current-match statistic is included: every rolling value uses shift(1).
roll_df = ordered[
    ['match_id', 'team_id', 'was_home', 'team_elo'] + roll_cols
]

home_roll = roll_df[roll_df.was_home == 1][
    ['match_id', 'team_elo'] + roll_cols
].rename(columns={
    'team_elo': 'home_elo',
    **{c: f'home_{c}' for c in roll_cols},
})

away_roll = roll_df[roll_df.was_home == 0][
    ['match_id', 'team_elo'] + roll_cols
].rename(columns={
    'team_elo': 'away_elo',
    **{c: f'away_{c}' for c in roll_cols},
})

match_features = home_roll.merge(
    away_roll,
    on='match_id',
    how='inner',
    validate='one_to_one',
)
match_features = match_features.merge(
    matches[['match_id', 'match_date', 'season_name', 'outcome']],
    on='match_id',
    how='inner',
    validate='one_to_one',
)

HOME_FORM_COLS = [f'home_roll_{c}' for c in safe_cols]
AWAY_FORM_COLS = [f'away_roll_{c}' for c in safe_cols]

HOME_FEAT_COLS = HOME_FORM_COLS + (['home_elo'] if USE_ELO else [])
AWAY_FEAT_COLS = AWAY_FORM_COLS + (['away_elo'] if USE_ELO else [])
FEAT_COLS = HOME_FEAT_COLS + AWAY_FEAT_COLS

match_features = (
    match_features
    .dropna(subset=FEAT_COLS)
    .sort_values('match_date')
    .reset_index(drop=True)
)

elo_text = 'with Elo' if USE_ELO else 'without Elo'
print(f'Final dataset: {len(match_features):,} matches × {len(FEAT_COLS)} features ({elo_text})')
print(f'Per side: {len(HOME_FEAT_COLS)} features')
print(match_features['outcome'].value_counts().to_string())

assert len(HOME_FEAT_COLS) == len(AWAY_FEAT_COLS)
assert not match_features[FEAT_COLS].isna().any().any()


## 6. Train/Test Split


In [ ]:
data = match_features.sort_values(['match_date', 'match_id']).reset_index(drop=True)

if SPLIT_MODE == 'last_season':
    train = data[data.season_name != TEST_SEASON].copy()
    test = data[data.season_name == TEST_SEASON].copy()
elif SPLIT_MODE == 'chronological_80_20':
    cut = int(len(data) * 0.80)
    train, test = data.iloc[:cut].copy(), data.iloc[cut:].copy()
elif SPLIT_MODE == 'stratified_80_20':
    train, test = train_test_split(
        data, test_size=0.20, random_state=SEED, stratify=data['outcome']
    )
else:
    raise ValueError('Unknown SPLIT_MODE')

if train.empty or test.empty:
    raise ValueError(
        f'Empty train/test split. Available seasons: {sorted(data.season_name.dropna().unique())}'
    )

X_train_raw = train[FEAT_COLS].to_numpy(dtype=np.float32)
X_test_raw = test[FEAT_COLS].to_numpy(dtype=np.float32)
y_train = train['outcome'].to_numpy()
y_test = test['outcome'].to_numpy()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

assert list(le.classes_) == ['A', 'D', 'H']
assert train[FEAT_COLS].columns.tolist() == test[FEAT_COLS].columns.tolist()

print(f'Train: {len(train):,} | {train.match_date.min().date()} → {train.match_date.max().date()}')
print(f'Test : {len(test):,} | {test.match_date.min().date()} → {test.match_date.max().date()}')
print(f'Features: {len(FEAT_COLS)} | classes: {list(le.classes_)}')
print('Train distribution:', pd.Series(y_train).value_counts().to_dict())
print('Test distribution :', pd.Series(y_test).value_counts().to_dict())


### 6.1 Distribution and Correlations


In [ ]:
_, axes = plt.subplots(1, 2, figsize=(15, 5))
classes_order = list(le.classes_)
train_pct = [(y_train == c).mean() * 100 for c in classes_order]
test_pct = [(y_test == c).mean() * 100 for c in classes_order]
x = np.arange(len(classes_order))
w = 0.35
axes[0].bar(x-w/2, train_pct, w, label='Train')
axes[0].bar(x+w/2, test_pct, w, label='Test')
axes[0].set_xticks(x, ['Away', 'Draw', 'Home'])
axes[0].set_ylabel('% matches')
axes[0].set_title('Class distribution')
axes[0].legend()

corr_df = train[HOME_FEAT_COLS].corr()
sns.heatmap(corr_df, cmap='coolwarm', center=0, ax=axes[1], cbar_kws={'label': 'correlation'})
axes[1].set_title('Home-side feature correlations')
axes[1].tick_params(axis='x', rotation=75, labelsize=7)
axes[1].tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.show()


## 7. Classical Models and Grid Search


In [ ]:
balanced = 'balanced' if USE_CLASS_WEIGHTS else None
equal_priors = (
    np.ones(len(le.classes_)) / len(le.classes_)
    if USE_CLASS_WEIGHTS else None
)

# Time-aware CV is used when the external split is chronological.
if SPLIT_MODE in {'last_season', 'chronological_80_20'}:
    classical_cv = TimeSeriesSplit(n_splits=GRID_CV_FOLDS)
    cv_name = f'TimeSeriesSplit({GRID_CV_FOLDS})'
else:
    classical_cv = StratifiedKFold(
        n_splits=GRID_CV_FOLDS,
        shuffle=True,
        random_state=SEED,
    )
    cv_name = f'StratifiedKFold({GRID_CV_FOLDS})'

print(f'Classical-model CV: {cv_name}')
print(f'Grid-search objective: {GRID_SEARCH_SCORING}')

# A pipeline is used so StandardScaler is fitted separately inside every
# cross-validation fold. Tree models do not need scaling, but scaling does
# not change their split ordering and lets all models use one safe workflow.
MODEL_SPECS = {
    'Random Forest': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('model', RandomForestClassifier(
                class_weight=balanced,
                random_state=SEED,
                n_jobs=1,
            )),
        ]),
        'grid': {
            'model__n_estimators': [150, 300, 500],
            'model__max_depth': [6, 8, 12, None],
            'model__min_samples_leaf': [1, 3],
        },
    },

    'Gaussian Naive Bayes': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('model', GaussianNB(priors=equal_priors)),
        ]),
        'grid': {
            'model__var_smoothing': np.logspace(-11, -7, 5),
        },
    },

    'SVM': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('model', SVC(
                kernel='rbf',
                class_weight=balanced,
            )),
        ]),
        # The paper explicitly states that C was tuned for SVM.
        'grid': {
            'model__C': [0.1, 0.3, 1.0, 3.0, 5.0, 10.0, 20.0],
            'model__gamma': ['scale'],
        },
    },

    'Decision Tree': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('model', DecisionTreeClassifier(
                class_weight=balanced,
                random_state=SEED,
            )),
        ]),
        'grid': {
            'model__max_depth': [3, 5, 6, 8, 12, None],
            'model__min_samples_leaf': [1, 3, 5],
        },
    },

    'KNN': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('model', KNeighborsClassifier(n_jobs=1)),
        ]),
        'grid': {
            'model__n_neighbors': [5, 7, 9, 11, 15, 21],
            'model__weights': ['uniform', 'distance'],
            'model__p': [1, 2],
        },
    },

    'Bagging': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('model', BaggingClassifier(
                estimator=DecisionTreeClassifier(
                    class_weight=balanced,
                    random_state=SEED,
                ),
                random_state=SEED,
                n_jobs=1,
            )),
        ]),
        'grid': {
            'model__n_estimators': [50, 100, 200, 300],
            'model__estimator__max_depth': [6, 10, None],
        },
    },
}

if HAS_XGBOOST:
    MODEL_SPECS['XGBoost'] = {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('model', XGBClassifier(
                random_state=SEED,
                eval_metric='mlogloss',
                n_jobs=1,
            )),
        ]),
        'grid': {
            'model__n_estimators': [200, 300],
            'model__learning_rate': [0.03, 0.05, 0.10],
            'model__max_depth': [3, 5],
            # The paper explicitly mentions gamma tuning for XGBoost.
            'model__gamma': [0.0, 0.1, 0.3],
        },
    }
else:
    print('XGBoost is not installed — the remaining models will still run.')

xgb_weights = (
    compute_sample_weight('balanced', y_train_enc)
    if USE_CLASS_WEIGHTS else None
)

def metric_row(name, y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=range(len(le.classes_)),
        zero_division=0,
    )
    idx = {c: i for i, c in enumerate(le.classes_)}
    d = idx['D']

    return {
        'Model': name,
        'Overall': 100 * accuracy_score(y_true, y_pred),
        'Balanced': 100 * balanced_accuracy_score(y_true, y_pred),
        'Macro F1': 100 * f1_score(y_true, y_pred, average='macro'),
        'Home Win': 100 * r[idx['H']],
        'Draw Precision': 100 * p[d],
        'Draw': 100 * r[d],
        'Draw F1': 100 * f[d],
        'Away Win': 100 * r[idx['A']],
        'Predicted Draws': int((y_pred == d).sum()),
    }

results = []
trained = {}
best_parameter_rows = []

for name, specification in MODEL_SPECS.items():
    estimator = specification['estimator']
    parameter_grid = specification['grid']

    fit_parameters = {}
    if name == 'XGBoost' and xgb_weights is not None:
        fit_parameters['model__sample_weight'] = xgb_weights

    if USE_GRID_SEARCH:
        search = GridSearchCV(
            estimator=estimator,
            param_grid=parameter_grid,
            scoring=GRID_SEARCH_SCORING,
            cv=classical_cv,
            n_jobs=-1,
            refit=True,
            return_train_score=False,
            error_score='raise',
        )

        search.fit(X_train_raw, y_train_enc, **fit_parameters)
        fitted_model = search.best_estimator_

        best_parameter_rows.append({
            'Model': name,
            'Best CV score': search.best_score_,
            'Best parameters': search.best_params_,
        })

        print(f'\n{name}')
        print(f'  best CV score: {search.best_score_:.4f}')
        print(f'  best parameters: {search.best_params_}')
    else:
        # Fixed fallback configuration: use the first value from each grid.
        fallback_params = {
            key: values[0]
            for key, values in parameter_grid.items()
        }
        fitted_model = estimator.set_params(**fallback_params)
        fitted_model.fit(X_train_raw, y_train_enc, **fit_parameters)

    prediction = fitted_model.predict(X_test_raw)

    trained[name] = (fitted_model, prediction)
    row = metric_row(name, y_test_enc, prediction)
    results.append(row)

    print(
        f"{name:24s} overall={row['Overall']:.1f}  "
        f"balanced={row['Balanced']:.1f}  "
        f"macroF1={row['Macro F1']:.1f}  "
        f"D(P/R/F1)="
        f"{row['Draw Precision']:.1f}/"
        f"{row['Draw']:.1f}/"
        f"{row['Draw F1']:.1f}  "
        f"predD={row['Predicted Draws']}"
    )

results_df = (
    pd.DataFrame(results)
    .round(1)
    .sort_values('Macro F1', ascending=False)
    .reset_index(drop=True)
)

display(results_df.style.hide(axis='index'))

if USE_GRID_SEARCH:
    best_parameters_df = pd.DataFrame(best_parameter_rows)
    display(best_parameters_df.style.hide(axis='index'))


### 7.1 Classical Model Visualisation


In [ ]:
metrics = ['Overall', 'Balanced', 'Macro F1', 'Draw F1']
plot_df = results_df.set_index('Model')[metrics]
_, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_df.plot(kind='bar', ax=axes[0], width=.8)
axes[0].set_ylabel('%')
axes[0].set_ylim(0, 100)
axes[0].set_title('Model quality')
axes[0].tick_params(axis='x', rotation=30)
sns.heatmap(plot_df, annot=True, fmt='.1f', cmap='YlGn', vmin=0, vmax=100, ax=axes[1])
axes[1].set_title('Metrics (%)')
plt.tight_layout()
plt.show()

best_name = results_df.iloc[0]['Model']
best_pred = trained[best_name][1]
cm = confusion_matrix(y_test_enc, best_pred, labels=range(len(le.classes_)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion matrix — {best_name}')
plt.show()


## 8. Autoencoder 1 — Reconstruction Error


In [ ]:
# ── Autoencoder architecture ──────────────────────────────────────────────────
class Autoencoder(nn.Module):
    """
    Symmetric encoder-decoder.
    Encoder: Input → 64 → 32 → bottleneck (latent_dim)
    Decoder: latent_dim → 32 → 64 → Input
    """
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)          # compress
        x_hat = self.decoder(z)      # reconstruct
        return x_hat

    def encode(self, x):
        return self.encoder(x)       # just the latent vector


def train_autoencoder(X_class, input_dim, latent_dim,
                      epochs=AE_EPOCHS, lr=AE_LR, batch_size=AE_BATCH):
    """Train one autoencoder on a single class's data."""
    model = Autoencoder(input_dim, latent_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    tensor = torch.tensor(X_class, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            recon = model(batch)
            loss  = criterion(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch + 1) % 20 == 0:
            print(f"    epoch {epoch+1:3d}/{epochs}  loss={epoch_loss/len(loader):.5f}")

    return model



def class_weights_from_labels(y_encoded):
    counts = np.bincount(y_encoded, minlength=len(le.classes_)).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def reconstruction_error(model, X):
    """Compute per-sample MSE reconstruction error."""
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        recon = model(t)
        errors = ((t - recon) ** 2).mean(dim=1).cpu().numpy()
    return errors


# ── Train one autoencoder per class ─────────────────────────────────────────
input_dim   = X_train.shape[1]
ae_models   = {}   # one per class label

print("Training class-specific autoencoders…\n")
for class_idx, class_name in enumerate(le.classes_):
    print(f"  Class '{class_name}' ({['Away Win','Draw','Home Win'][class_idx]}):")
    mask = y_train_enc == class_idx
    X_class = X_train[mask]
    print(f"    Training samples: {len(X_class)}")
    ae = train_autoencoder(X_class, input_dim, LATENT_DIM)
    ae_models[class_idx] = ae
    print()

print("All autoencoders trained.")


In [ ]:
# ── Predict using reconstruction errors ──────────────────────────────────────
# For each test sample, compute reconstruction error under each class's AE.
# Predict the class with the LOWEST reconstruction error.

error_matrix = np.zeros((len(X_test), len(le.classes_)))

for class_idx in range(len(le.classes_)):
    errors = reconstruction_error(ae_models[class_idx], X_test)
    error_matrix[:, class_idx] = errors

# Predict = class with minimum reconstruction error
y_pred_ae1 = np.argmin(error_matrix, axis=1)

# ── Metrics ───────────────────────────────────────────────────────────────────
overall_ae1 = accuracy_score(y_test_enc, y_pred_ae1) * 100
cm_ae1 = confusion_matrix(y_test_enc, y_pred_ae1, labels=range(len(le.classes_)))
per_class_ae1 = cm_ae1.diagonal() / cm_ae1.sum(axis=1).clip(min=1) * 100
class_acc_ae1 = dict(zip(le.classes_, per_class_ae1))

print("=" * 55)
print("AUTOENCODER 1 — Reconstruction Error Results")
print("=" * 55)
print(f"  Overall accuracy : {overall_ae1:.1f}%")
print(f"  Home Win accuracy: {class_acc_ae1.get('H', 0):.1f}%")
print(f"  Draw accuracy    : {class_acc_ae1.get('D', 0):.1f}%")
print(f"  Away Win accuracy: {class_acc_ae1.get('A', 0):.1f}%")
print()
print(classification_report(y_test_enc, y_pred_ae1, target_names=le.classes_))

# ── Visualise reconstruction errors ──────────────────────────────────────────
_, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#E91E63', '#FF9800', '#4CAF50']

for true_class in range(len(le.classes_)):
    mask = y_test_enc == true_class
    for pred_class in range(len(le.classes_)):
        axes[true_class].hist(
            error_matrix[mask, pred_class], bins=30, alpha=0.6,
            label=f'AE_{le.classes_[pred_class]}', color=colors[pred_class]
        )
    axes[true_class].set_title(f'True class: {le.classes_[true_class]}')
    axes[true_class].set_xlabel('Reconstruction Error')
    axes[true_class].set_ylabel('Count')
    axes[true_class].legend(fontsize=8)

plt.suptitle('Reconstruction Errors per True Class\n'
             '(lower error = model recognises the pattern)', fontsize=11)
plt.tight_layout()
plt.savefig('ae1_reconstruction_errors.png', dpi=120, bbox_inches='tight')
plt.show()


## 9. Autoencoder 2 — Latent-Space Classifier


In [ ]:
# ── Stage 1: Train a single autoencoder on ALL training data ─────────────────
print("Stage 1 — Training autoencoder on all training data…")
ae_full = train_autoencoder(X_train, input_dim, LATENT_DIM,
                             epochs=AE_EPOCHS, lr=AE_LR, batch_size=AE_BATCH)
print("Autoencoder trained.\n")

# ── Stage 2: Extract latent representations ───────────────────────────────────
def get_latent(model, X):
    """Pass X through encoder only → latent vectors."""
    model.eval()
    with torch.no_grad():
        t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        z = model.encode(t).cpu().numpy()
    return z

Z_train = get_latent(ae_full, X_train)
Z_test  = get_latent(ae_full, X_test)

print(f"Latent space shape — Train: {Z_train.shape}  Test: {Z_test.shape}")
print(f"Compressed from {input_dim} features → {LATENT_DIM} latent dimensions")
print(f"Compression ratio: {input_dim/LATENT_DIM:.1f}×")


In [ ]:
# ── Stage 3: Classifier network on the latent space ──────────────────────────
class LatentClassifier(nn.Module):
    """
    Small MLP that sits on top of the latent space.
    Latent (16) → 32 → 16 → 3 classes (H/D/A)
    """
    def __init__(self, latent_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes),
        )

    def forward(self, z):
        return self.net(z)


n_classes  = len(le.classes_)
classifier = LatentClassifier(LATENT_DIM, n_classes).to(DEVICE)
optimizer  = optim.Adam(classifier.parameters(), lr=AE_LR)
criterion = nn.CrossEntropyLoss(
    weight=class_weights_from_labels(y_train_enc) if USE_CLASS_WEIGHTS else None
)

# Build DataLoader
Z_t = torch.tensor(Z_train, dtype=torch.float32)
y_t = torch.tensor(y_train_enc, dtype=torch.long)
loader_clf = DataLoader(TensorDataset(Z_t, y_t), batch_size=AE_BATCH, shuffle=True)

print("Stage 3 — Training latent-space classifier…")
for epoch in range(AE_EPOCHS):
    classifier.train()
    epoch_loss = 0.0
    for (zb, yb) in loader_clf:
        zb, yb = zb.to(DEVICE), yb.to(DEVICE)
        logits = classifier(zb)
        loss   = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"    epoch {epoch+1:3d}/{AE_EPOCHS}  loss={epoch_loss/len(loader_clf):.4f}")

print("Classifier trained.")


In [ ]:
# ── Evaluate Autoencoder 2 ────────────────────────────────────────────────────
classifier.eval()
with torch.no_grad():
    Z_te = torch.tensor(Z_test, dtype=torch.float32).to(DEVICE)
    logits = classifier(Z_te)
    y_pred_ae2 = logits.argmax(dim=1).cpu().numpy()

overall_ae2 = accuracy_score(y_test_enc, y_pred_ae2) * 100
cm_ae2 = confusion_matrix(y_test_enc, y_pred_ae2, labels=range(n_classes))
per_class_ae2 = cm_ae2.diagonal() / cm_ae2.sum(axis=1).clip(min=1) * 100
class_acc_ae2 = dict(zip(le.classes_, per_class_ae2))

print("=" * 55)
print("AUTOENCODER 2 — Latent Space Classifier Results")
print("=" * 55)
print(f"  Overall accuracy : {overall_ae2:.1f}%")
print(f"  Home Win accuracy: {class_acc_ae2.get('H', 0):.1f}%")
print(f"  Draw accuracy    : {class_acc_ae2.get('D', 0):.1f}%")
print(f"  Away Win accuracy: {class_acc_ae2.get('A', 0):.1f}%")
print()
print(classification_report(y_test_enc, y_pred_ae2, target_names=le.classes_))


### 9.1 Latent-Space Visualisation


In [ ]:
# ── Visualise the latent space with t-SNE ────────────────────────────────────
print("Running t-SNE on latent space (may take ~30s)…")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
Z_2d = tsne.fit_transform(Z_test)

colors_map = {'A': '#E91E63', 'D': '#FF9800', 'H': '#4CAF50'}
_, axes = plt.subplots(1, 2, figsize=(14, 5))

# t-SNE scatter — true labels
for cls in le.classes_:
    mask = y_test == cls
    axes[0].scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                    c=colors_map[cls], label=cls, alpha=0.5, s=15)
axes[0].set_title('Latent Space (t-SNE) — True Labels')
axes[0].legend(title='Outcome')
axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')

# t-SNE scatter — predicted labels
pred_labels = le.inverse_transform(y_pred_ae2)
for cls in le.classes_:
    mask = pred_labels == cls
    axes[1].scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                    c=colors_map[cls], label=cls, alpha=0.5, s=15)
axes[1].set_title('Latent Space (t-SNE) — Predicted Labels')
axes[1].legend(title='Predicted')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')

plt.suptitle('Autoencoder 2 — Latent Space Visualisation', fontsize=12)
plt.tight_layout()
plt.savefig('ae2_latent_space.png', dpi=120, bbox_inches='tight')
plt.show()


## 10. Supervised Neural Models


In [ ]:
# Shared helpers for deep models
extra_results = []

def add_result(model_name, y_true_enc, y_pred_enc):
    row = metric_row(model_name, y_true_enc, y_pred_enc)
    extra_results.append(row)
    print(
        f"{model_name:28s} overall={row['Overall']:.1f}  balanced={row['Balanced']:.1f}  "
        f"macroF1={row['Macro F1']:.1f}  D(P/R/F1)="
        f"{row['Draw Precision']:.1f}/{row['Draw']:.1f}/{row['Draw F1']:.1f}  "
        f"predD={row['Predicted Draws']}"
    )

def train_torch_classifier(model, loader, epochs=50, lr=1e-3, weight=None, grad_clip=1.0):
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weight)
    for epoch in range(epochs):
        model.train(); epoch_loss = 0.0
        for batch in loader:
            *inputs, yb = batch
            inputs = [x.to(DEVICE) for x in inputs]; yb = yb.to(DEVICE)
            logits = model(*inputs); loss = criterion(logits, yb)
            optimizer.zero_grad(); loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); epoch_loss += loss.item()
        if (epoch + 1) % 20 == 0:
            print(f'  epoch {epoch+1:3d}/{epochs} loss={epoch_loss/len(loader):.4f}')
    return model

def predict_torch_classifier(model, *inputs):
    model.eval()
    with torch.no_grad():
        logits = model(*[x.to(DEVICE) for x in inputs])
        return logits.argmax(dim=1).cpu().numpy()

def split_home_away(X):
    half = X.shape[1] // 2
    assert X.shape[1] % 2 == 0
    return X[:, :half], X[:, half:]


### 10.1 Multilayer Perceptron


In [ ]:
class MatchMLP(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.net(x)

mlp = MatchMLP(X_train.shape[1], len(le.classes_))
mlp_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train_enc, dtype=torch.long),
    ), batch_size=64, shuffle=True,
)
mlp = train_torch_classifier(
    mlp, mlp_loader, epochs=70, lr=1e-3,
    weight=class_weights_from_labels(y_train_enc) if USE_CLASS_WEIGHTS else None,
)
y_pred_mlp = predict_torch_classifier(mlp, torch.tensor(X_test, dtype=torch.float32))
add_result('Multilayer Perceptron', y_test_enc, y_pred_mlp)
print(classification_report(y_test_enc, y_pred_mlp, target_names=le.classes_, zero_division=0))


### 10.2 Siamese Network


In [ ]:
# Siamese architecture aligned with Table 1 of the paper:
# twin hidden=128, head hidden=64, batch=128, learning rate=1e-4, CCE.
half_dim = X_train.shape[1] // 2
Xh_train, Xa_train = split_home_away(X_train)
Xh_test, Xa_test = split_home_away(X_test)

class SiameseMatchNet(nn.Module):
    def __init__(self, input_dim, emb_dim=128, head_dim=64, n_classes=3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, emb_dim), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(emb_dim * 2, head_dim), nn.ReLU(),
            nn.Linear(head_dim, n_classes),
        )
    def forward(self, home_x, away_x):
        h = self.encoder(home_x)
        a = self.encoder(away_x)
        return self.head(torch.cat([h, a], dim=1))

siamese = SiameseMatchNet(half_dim, emb_dim=128, head_dim=64, n_classes=len(le.classes_))
siamese_loader = DataLoader(
    TensorDataset(
        torch.tensor(Xh_train, dtype=torch.float32),
        torch.tensor(Xa_train, dtype=torch.float32),
        torch.tensor(y_train_enc, dtype=torch.long),
    ), batch_size=128, shuffle=True,
)

# Keep weight=None for a faithful paper-style architecture. Run a separately
# named weighted variant if you want to test the effect of class weighting.
siamese = train_torch_classifier(siamese, siamese_loader, epochs=70, lr=1e-4, weight=None)
y_pred_siamese = predict_torch_classifier(
    siamese,
    torch.tensor(Xh_test, dtype=torch.float32),
    torch.tensor(Xa_test, dtype=torch.float32),
)
add_result('Siamese Network', y_test_enc, y_pred_siamese)
print(classification_report(y_test_enc, y_pred_siamese, target_names=le.classes_, zero_division=0))


## 11. Full Model Comparison


In [ ]:
# Final comparison with metrics that are meaningful under class imbalance.
all_results = pd.concat(
    [results_df, pd.DataFrame(extra_results)], ignore_index=True, sort=False
)

# Add autoencoder results if their cells were executed.
if 'y_pred_ae1' in globals():
    all_results = pd.concat([
        all_results,
        pd.DataFrame([metric_row('AE1 — Reconstruction Error', y_test_enc, y_pred_ae1)])
    ], ignore_index=True)
if 'y_pred_ae2' in globals():
    all_results = pd.concat([
        all_results,
        pd.DataFrame([metric_row('AE2 — Latent Classifier', y_test_enc, y_pred_ae2)])
    ], ignore_index=True)

all_results = all_results.round(1).sort_values('Macro F1', ascending=False).reset_index(drop=True)
display(all_results)

metrics = ['Overall', 'Balanced', 'Macro F1', 'Draw F1']
heat = all_results.set_index('Model')[metrics]
_, axes = plt.subplots(1, 2, figsize=(17, max(6, .42*len(all_results))))
heat.plot(kind='bar', ax=axes[0], width=.82)
axes[0].set_ylim(0, 100); axes[0].set_ylabel('%'); axes[0].set_title('All models')
axes[0].tick_params(axis='x', rotation=35)
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlGn', vmin=0, vmax=100, ax=axes[1])
axes[1].set_title('Accuracy, balance, macro-F1 and draw-F1')
plt.tight_layout(); plt.show()
